In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "db" / "retail.db"

print("Project root:", PROJECT_ROOT)
print("CSV files:", [f.name for f in RAW_DIR.glob("*.csv")])

Project root: C:\Users\samee\Desktop\retail-project
CSV files: ['customers.csv', 'orders.csv', 'order_items.csv', 'products.csv', 'stores.csv', 'transactions.csv']


In [2]:
stores = pd.read_csv(RAW_DIR / "stores.csv")

print("Shape (rows, columns):", stores.shape)
stores.head()

Shape (rows, columns): (12, 6)


,store_id,store_name,city,region,store_type,opened_date
0,S001,Bengaluru Express,Bengaluru,South,Express,2022-03-18
1,S002,Hyderabad Standard,Hyderabad,South,Standard,2021-08-23
2,S003,Chennai Express,Chennai,South,Express,2020-08-26
3,S004,Mumbai Standard,Mumbai,West,Standard,2022-05-19
4,S005,Pune Flagship,Pune,West,Flagship,2021-03-29


In [3]:
orders = pd.read_csv(RAW_DIR / "orders.csv", dtype=str, keep_default_na=False)

print("Shape:", orders.shape)
orders.head()

Shape: (82089, 6)


,order_id,customer_id,store_id,order_ts,channel,status
0,O0000001,C01849,S004,2024-01-02 16:34:00,Online,Completed
1,O0000002,C02181,S002,2024-01-03 11:55:00,In-store,Completed
2,O0000003,C02489,S005,2024-01-03 18:32:00,In-store,Completed
3,O0000004,C00570,S011,2024-01-04 19:20:00,Online,Completed
4,O0000005,C00521,S009,2024-01-05 16:30:00,Online,Completed


In [4]:
good_pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"

bad_dates = orders[~orders["order_ts"].str.match(good_pattern)]

print("Rows with a badly formatted date:", len(bad_dates))
bad_dates["order_ts"].value_counts()

Rows with a badly formatted date: 15


order_ts
31/02/2025    8
              7
Name: count, dtype: int64

In [8]:
import sqlite3
from datetime import datetime, timezone

stores = pd.read_csv(RAW_DIR / "stores.csv", dtype=str, keep_default_na=False)

stores["_source_file"] = "stores.csv"
stores["_loaded_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

conn = sqlite3.connect(DB_PATH)
rows = stores.to_sql("raw_stores", conn, if_exists="replace", index=False)
print("Rows saved:", rows)

pd.read_sql("SELECT * FROM raw_stores LIMIT 3", conn)

Rows saved: 12


,store_id,store_name,city,region,store_type,opened_date,_source_file,_loaded_at
0,S001,Bengaluru Express,Bengaluru,South,Express,2022-03-18,stores.csv,2026-10-26T09:43:50+00:00
1,S002,Hyderabad Standard,Hyderabad,South,Standard,2021-08-23,stores.csv,2026-10-26T09:43:50+00:00
2,S003,Chennai Express,Chennai,South,Express,2020-08-26,stores.csv,2026-10-26T09:43:50+00:00


In [9]:
EXPECTED_COLUMNS = {
    "stores": ["store_id", "store_name", "city", "region", "store_type", "opened_date"],
    "products": ["product_id", "product_name", "category", "subcategory", "list_price", "unit_cost"],
    "customers": ["customer_id", "signup_date", "city", "segment", "birth_year"],
    "orders": ["order_id", "customer_id", "store_id", "order_ts", "channel", "status"],
    "order_items": ["order_item_id", "order_id", "product_id", "quantity", "unit_price",
                    "discount_pct", "is_returned"],
    "transactions": ["transaction_id", "order_id", "payment_method", "amount",
                     "transaction_ts", "status"],
}

print("Tables to load:", list(EXPECTED_COLUMNS))

Tables to load: ['stores', 'products', 'customers', 'orders', 'order_items', 'transactions']


In [10]:
def load_table(name, expected_columns, conn):
    path = RAW_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    missing = set(expected_columns) - set(df.columns)
    if missing:
        raise ValueError(f"{name}.csv is missing columns: {sorted(missing)}")

    df["_source_file"] = path.name
    df["_loaded_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    df.to_sql(f"raw_{name}", conn, if_exists="replace", index=False)
    return len(df)

In [11]:
for name, columns in EXPECTED_COLUMNS.items():
    row_count = load_table(name, columns, conn)
    print(f"raw_{name:<15} {row_count:>8,} rows")

raw_stores                12 rows
raw_products             300 rows
raw_customers          4,015 rows
raw_orders            82,089 rows
raw_order_items      204,230 rows
raw_transactions      84,390 rows


In [12]:
try:
    load_table("stores", EXPECTED_COLUMNS["stores"] + ["manager_name"], conn)
except ValueError as error:
    print("Caught the problem:", error)

Caught the problem: stores.csv is missing columns: ['manager_name']


In [13]:
def sql(query):
    return pd.read_sql(query, conn)

sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")

,name
0,raw_customers
1,raw_order_items
2,raw_orders
3,raw_products
4,raw_stores
5,raw_transactions


In [14]:
sql("""
SELECT s.region, COUNT(*) AS orders
FROM raw_orders AS o
JOIN raw_stores AS s ON s.store_id = o.store_id
GROUP BY s.region
ORDER BY orders DESC
""")

,region,orders
0,West,23652
1,North,22928
2,East,19289
3,South,16220


In [15]:
sql("""
SELECT p.category,
       ROUND(SUM(CAST(oi.quantity AS REAL) * CAST(oi.unit_price AS REAL))) AS gross_sales
FROM raw_order_items AS oi
JOIN raw_products AS p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY gross_sales DESC
""")

,category,gross_sales
0,Electronics,219945790.0
1,Fashion,122157340.0
2,Home&Kitchen,111330310.0
3,Home & Kitchen,97265170.0
4,electronics,73356960.0
5,Sports,62353190.0
6,Electronics,21676630.0
7,Grocery,15492540.0
8,fashion,8139830.0
9,Apparel,5534840.0


In [16]:
conn.close()
print("Connection closed")

Connection closed
